# Notebook 2 — Spatial Grid and Time Series

This notebook builds the spatial grid of ~200 m × 200 m cells, aggregates theft events per cell, constructs hourly binary event time series, and briefly characterises the series before the Event Synchronization step.

> **Thesis parameters:** `grid_size = 0.002°` (~200 m), `min_events = 10` non-zero hours.
Adjust these in `config.py` or pass different values to the functions below.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from vehicle_theft_network.config import DataConfig, GridConfig, TimeSeriesConfig
from vehicle_theft_network.data.loader import load_pickle, save_pickle, save_npy
from vehicle_theft_network.grid.builder import build_grid_cells, aggregate_cells
from vehicle_theft_network.timeseries.builder import (
    build_time_series, filter_time_series, to_binary_event_series
)

data_cfg   = DataConfig()
grid_cfg   = GridConfig()
ts_cfg     = TimeSeriesConfig()

## 2. Load Preprocessed Data

In [ ]:
dados = pd.read_parquet(data_cfg.processed_parquet)
dados['DATA_HORA'] = pd.to_datetime(dados.get('DATA_HORA', pd.NaT), errors='coerce')
print(f'{len(dados):,} records loaded')
dados.head(3)

## 3. Spatial Grid

Each theft is assigned to a `grid_size × grid_size` degree cell. The cell is represented by a square polygon whose corners are defined by the floor-divided lat/lon indices.

**Thesis value:** `grid_size = 0.002°` ≈ 200 m × 200 m at São Paulo's latitude.

In [1]:
grid_cells = build_grid_cells(dados, grid_size=grid_cfg.grid_size)
print(f'{len(grid_cells):,} rows (one per theft, one polygon per row)')
print(f'{grid_cells["geometry"].nunique():,} unique grid cells')
grid_cells.head(3)

276,503 rows (one per theft, one polygon per row)
68,384 unique grid cells


### Cell area verification (UTM Zone 23S)

In [2]:
grid_copy = grid_cells.copy().set_crs('EPSG:4326').to_crs('EPSG:32723')
areas_km2 = grid_copy['geometry'].area / 1e6
print(f'Mean cell area: {areas_km2.mean():.4f} km²  (~{areas_km2.mean()*1e6:.0f} m²)')
areas_km2.head()

Mean cell area: 0.0452 km²  (~45200 m²)


0         0.045224
1         0.045229
2         0.045226
...
276498    0.045182
Length: 276503, dtype: float64


## 4. Cell Aggregation

Events are grouped by cell centroid. The result is a dictionary keyed by `(longitude, latitude)` of the centroid, holding the list of event timestamps and the sets of city/neighbourhood names observed in that cell.

In [3]:
cell_data_dict = aggregate_cells(grid_cells)
print(f'{len(cell_data_dict):,} unique cells after aggregation')

68,384 unique cells after aggregation


## 5. Hourly Time Series

For each cell, events are counted per hour to produce a multivariate time series of shape `(T, N)` where T is the number of hours in the study period and N is the number of cells.

In [ ]:
ts_df = build_time_series(cell_data_dict, freq=ts_cfg.frequency)
print(f'Time series shape: {ts_df.shape}  (hours × cells)')
ts_df.head()

### 5.1 Filter by minimum event count

Cells with fewer than `min_events` non-zero hours are discarded; they carry too little signal for a meaningful synchronization estimate.

**Thesis value:** `min_events = 10`

In [4]:
ts_filtered = filter_time_series(ts_df, min_events=ts_cfg.min_events)
print(f'{ts_df.shape[1]:,} cells before filter  ->  {ts_filtered.shape[1]:,} retained')

68,384 cells before filter  ->  1,336 retained


### 5.2 Convert to binary event series

In [5]:
event_array = to_binary_event_series(ts_filtered)
print(f'event_array shape: {event_array.shape}  (hours × retained cells)')
print(f'Fraction of non-zero entries: {event_array.mean():.4f}')

event_array shape: (17520, 1336)  (hours × retained cells)
Fraction of non-zero entries: 0.0040


## 6. Exploratory Analysis of Event Series

### 6.1 Pairing coefficient

The pairing coefficient for a series measures the fraction of consecutive pairs of 1s (events on back-to-back hours) relative to the total number of events.

In [ ]:
def pairing_coefficient(series_col):
    count = ((series_col[:-1] == 1) & (series_col[1:] == 1)).sum()
    total = (series_col == 1).sum()
    return count / (total - 1) if total > 1 else 0.0

pairing = {col: pairing_coefficient(event_array[:, col]) for col in range(event_array.shape[1])}
import numpy as np
vals = list(pairing.values())
print(f'Pairing coefficient — mean: {np.mean(vals):.4f}  max: {np.max(vals):.4f}  ')
print(f'                       Q50: {np.percentile(vals,50):.4f}  Q95: {np.percentile(vals,95):.4f}')

### 6.2 Grouping rate

Complement of the fraction of isolated events (events not adjacent to any other event). Higher values indicate more clustered (bursty) activity patterns.

In [ ]:
def grouping_rate(series_col):
    ones = np.where(series_col == 1)[0]
    isolated = sum(
        1 for i in ones
        if (i == 0 or series_col[i-1] == 0) and
           (i == len(series_col)-1 or series_col[i+1] == 0)
    )
    total = len(ones)
    return 1 - isolated / total if total > 0 else 0.0

grouping = {col: grouping_rate(event_array[:, col]) for col in range(event_array.shape[1])}
gvals = list(grouping.values())
print(f'Grouping rate — mean: {np.mean(gvals):.4f}  max: {np.max(gvals):.4f}')

## 7. Monte Carlo Significance Test

The Event Synchronization significance test compares the observed pairwise ES score against a null distribution of 1 000 random surrogates (shuffled event positions). A pair is considered significant at the 95% level if its score exceeds that of more than 950 surrogates.

This computation is **very CPU-intensive** (hours on a single machine for 1 336 cells). The original thesis used an HPC cluster. The code below is provided for reference; the pre-computed result is loaded in Notebook 3.

```python
from vehicle_theft_network.event_sync.analyzer import (
    build_event_series, compute_significance
)
ev = build_event_series(event_array, taumax=5)
# This call takes hours — run on a cluster
monte_carlo_counts = compute_significance(ev, n_surr=1000, symmetrization='symmetric')
np.save('monteCarlosimulations.npy', monte_carlo_counts)
```

## 8. Save Intermediate Data

In [ ]:
ts_filtered.to_pickle(data_cfg.time_series_path)
save_pickle(grid_cells, data_cfg.grid_cells_path)
save_pickle(cell_data_dict, data_cfg.cell_data_dict_path)
save_npy(event_array, str(data_cfg.monte_carlo_path).replace('monteCarlosimulations', 'event_series'))
print('Intermediate data saved.')